# Volume 12, Chapter 14: Modular forms and $L$-functions (Python companion)

Exercises first, solutions at the end. We build $q$-expansions with exact integers, invert the $j$-function, diagonalise the Hecke operator $T_2$ on $S_{24}$, compute $L(\Delta,s)$ from Hecke's integral formula and find its first zeros, match the coefficients of $\eta(\tau)^2\eta(11\tau)^2$ with point counts on $y^2+y=x^3-x^2$ and test the Birch–Swinnerton-Dyer formula, and finish with the algorithm behind most record computations of $\pi$: **Chudnovsky's series with binary splitting**. Requires `cypari2` and `mpmath`.

In [ ]:
import math, time, sys
sys.set_int_max_str_digits(0)
from fractions import Fraction as Fr
import mpmath
import cypari2
pari = cypari2.Pari()
pari.allocatemem(2 * 10**9, silent=True)

## Exercises

**P14.1 (★) $q$-expansions and $j$.** Write power-series helpers (lists of integers, truncated at $q^{M}$) and build $E_4,E_6,E_8,\Delta=q\prod(1-q^n)^{24}$ and $j=E_4^3/\Delta$. Check $E_4^2=E_8$ and $1728\Delta=E_4^3-E_6^2$ (Volume 5) and the integrality of $j$ (Exercise 14.5). Evaluate $j$ at $i$ and $\rho$ from the $q$-series. Then test Theorem 14.4 backwards: for a few complex numbers $c$, find $\tau\in F$ with $j(\tau)=c$ (PARI: `ellfromj`, `ellperiods`), and check by evaluating the series.

**P14.2 (★★) Hecke operators.** Implement $T_n$ on $q$-expansions by the formula of Theorem 14.6(a). Check $T_2T_3=T_6$ and $T_2T_4=T_8+2^{k-1}T_2$ on $\Delta E_4^3$ (weight 24). Compute the matrix of $T_2$ on $S_{24}$ in the basis $\Delta^2,\ \Delta E_4^3$, its eigenvalues $540\pm12\sqrt{144169}$, and the two normalised eigenforms; check $c_6=c_2c_3$ for both.

**P14.3 (★★) $L(\Delta,s)$.** From the proof of Theorem 14.9 with $N=1$, $k=12$: $\Lambda(s)=I(s)+I(12-s)$ with $I(s)=\sum_n\tau(n)(2\pi n)^{-s}\Gamma(s,2\pi n)$ (upper incomplete gamma). Implement it with `mpmath`, check $\Lambda(s)=\Lambda(12-s)$ at a random complex $s$, compute $L(\Delta,6)$ and compare with PARI, and locate the first zeros of $L(\Delta,6+it)$ by sign changes of $Z(t)$ (Figure 14.2).

**P14.4 (★★) The curve of conductor 11.** Compare the coefficients $c_p$ of $\eta(\tau)^2\eta(11\tau)^2$ with $a_p=p+1-N_p$ for $E:y^2+y=x^3-x^2$ for all $p<10^5$. Prove from Theorem 14.9 that $L(f,1)=2\sum_{n\ge1}\frac{c_n}ne^{-2\pi n/\sqrt{11}}$, evaluate it, and compute $L(f,1)/\Omega$ with $\Omega$ the real period of $E$. Compare with the Birch–Swinnerton-Dyer prediction $\frac{|\text{Ш}|\prod c_v}{|E(\mathbb Q)_{\rm tors}|^2}$ for $E(\mathbb Q)\cong\mathbb Z/5$.

**P14.5 (★★★) CS application: $\pi$ by Chudnovsky and binary splitting.** The Chudnovsky series
$$\frac1\pi=12\sum_{k\ge0}\frac{(-1)^k(6k)!\,(13591409+545140134k)}{(3k)!\,(k!)^3\,640320^{3k+3/2}}$$
gains about $14$ digits per term; it comes from $j\bigl(\frac{1+\sqrt{-163}}2\bigr)=-640320^3$ (Remark 14.14). Summing term by term with big rationals costs $O(n^2)$; *binary splitting* computes the sum of terms $a..b$ as an exact fraction $P(a,b),Q(a,b),T(a,b)$ by recursion on halves, so that all the big multiplications happen between numbers of similar size, and the cost drops to $O(M(n)\log^2n)$ with $M$ the cost of multiplication. Implement it, compute $10^5$ digits of $\pi$, compare with `mpmath`, and time it. Check numerically that $e^{\pi\sqrt{163}}$ is within $10^{-12}$ of an integer.

## Solutions

### P14.1
**How to find it.** Series are integer lists; $1/\Delta$ is $q^{-1}$ times the inverse of $\prod(1-q^n)^{24}$, computed coefficient by coefficient. Evaluating the series at $\tau=i$ uses $q=e^{-2\pi}\approx0.0019$, so thirty terms give full double precision.

In [ ]:
M = 40
def mul(a, b):
    c = [0] * M
    for i, x in enumerate(a):
        if x:
            for j in range(M - i): c[i + j] += x * b[j]
    return c
def inv(a):                      # a[0] = 1
    b = [0] * M; b[0] = 1
    for n in range(1, M): b[n] = -sum(a[i] * b[n - i] for i in range(1, n + 1))
    return b
sigma = lambda n, k: sum(d**k for d in range(1, n + 1) if n % d == 0)
E4 = [1] + [240 * sigma(n, 3) for n in range(1, M)]
E6 = [1] + [-504 * sigma(n, 5) for n in range(1, M)]
E8 = [1] + [480 * sigma(n, 7) for n in range(1, M)]
P = [1] + [0] * (M - 1)
for n in range(1, M):
    f = [0] * M; f[0] = 1; f[n] = -1
    for _ in range(24): P = mul(P, f)
Delta = [0] + P[:M - 1]                         # q * prod(1-q^n)^24
print("E4^2 = E8:", mul(E4, E4) == E8)
print("E4^3 - E6^2 = 1728 Delta:", [x - y for x, y in zip(mul(mul(E4, E4), E4), mul(E6, E6))] == [1728 * d for d in Delta])
jser = mul(mul(mul(E4, E4), E4), inv(P))         # j = q^{-1} * jser
print("tau(n):", Delta[1:9])
print("j = q^-1 +", jser[1], "+", jser[2], "q +", jser[3], "q^2 + ...  (all integers:", all(isinstance(x, int) for x in jser), ")")

def j_eval(tau):
    q = complex(math.e**(-2 * math.pi * tau.imag)) * complex(math.cos(2 * math.pi * tau.real), math.sin(2 * math.pi * tau.real))
    return sum(c * q**(n - 1) for n, c in enumerate(jser))
rho = complex(-0.5, math.sqrt(3) / 2)
print("j(i) =", j_eval(1j), "  j(rho) =", j_eval(rho))

In [ ]:
def tau_from_j(c):
    E = pari.ellinit(pari.ellfromj(c))
    w1, w2 = pari.ellperiods(E)
    t = complex(w1 / w2)
    if t.imag < 0: t = -t
    # reduce to the fundamental domain (Volume 5, Theorem 12.2)
    while True:
        t = complex(t.real - round(t.real), t.imag)
        if abs(t) < 1 - 1e-12: t = -1 / t
        else: return t
for c in (0, 1728, 1, -1, 2 + 3j, 10**6):
    t = tau_from_j(pari(c) if not isinstance(c, complex) else pari(c.real) + pari(c.imag) * pari('I'))
    print(f"c = {c!s:>8}: tau = {t.real:+.6f}{t.imag:+.6f}i   j(tau) = {j_eval(t):.6f}")

Every value is taken, once, on the fundamental domain: $c=0$ at $\rho$, $c=1728$ at $i$, real $c$ on the boundary or on the imaginary axis (Exercise 14.1), and $c=10^6$ high up, where $j\approx q^{-1}$. The integer $196884$ is the coefficient of $q$.

### P14.2
**How to find it.** $c_m(T_nf)=\sum_{a\mid(m,n)}a^{k-1}c_{mn/a^2}$ needs the coefficients of $f$ up to $mn$, so we keep more terms. In the basis $\Delta^2=q^2+\cdots$, $g=\Delta E_4^3=q+\cdots$, a cusp form of weight 24 is determined by its coefficients of $q$ and $q^2$.

In [ ]:
M = 130
E4 = [1] + [240 * sigma(n, 3) for n in range(1, M)]
Delta = [0] + [int(x) for x in pari(f'Vec(eta(q + O(q^{M-1}))^24)')]
D2 = mul(Delta, Delta); G = mul(Delta, mul(E4, mul(E4, E4)))

def T(n, f, k, terms):
    return [sum(a**(k - 1) * f[m * n // a**2] for a in range(1, min(m, n) + 1) if m % a == 0 and n % a == 0) if m else
            sum(a**(k - 1) for a in range(1, n + 1) if n % a == 0) * f[0] for m in range(terms)]

k = 24
lhs = T(2, T(3, G, k, 30), k, 12); rhs = T(6, G, k, 12)
print("T2 T3 = T6 on Delta*E4^3:", lhs == rhs)
lhs = T(2, T(4, G, k, 30), k, 12); rhs = [x + 2**(k - 1) * y for x, y in zip(T(8, G, k, 12), T(2, G, k, 12))]
print("T2 T4 = T8 + 2^23 T2:", lhs == rhs)

def coords(f):                     # f = x*G + y*D2, read off from q^1 and q^2
    x = f[1]; y = f[2] - x * G[2]
    return x, y
cols = [coords(T(2, D2, k, 3)), coords(T(2, G, k, 3))]   # images of D2 and G
A = [[cols[1][0], cols[0][0]], [cols[1][1], cols[0][1]]] # matrix in basis (G, D2)
tr = A[0][0] + A[1][1]; det = A[0][0] * A[1][1] - A[0][1] * A[1][0]
print("matrix of T2 in basis (Delta E4^3, Delta^2):", A)
print("characteristic polynomial: x^2 -", tr, "x +", det, "  discriminant =", pari.factor(tr * tr - 4 * det))

In [ ]:
# eigenvalues 540 +- 12 sqrt(144169); eigenvectors in Q(sqrt d) represented numerically with high precision
mpmath.mp.dps = 40
d = 144169
for sgn in (1, -1):
    lam = 540 + sgn * 12 * mpmath.sqrt(d)
    # eigenvector (x, y) of A with x = 1: (A00 - lam) + A01 y = 0
    y = (lam - A[0][0]) / A[0][1]
    c = [G[n] + y * D2[n] for n in range(12)]                 # normalised: c1 = 1
    print(f"lambda_2 = {mpmath.nstr(lam, 15)}:  c2 = {mpmath.nstr(c[2], 15)}, c6 - c2 c3 = {mpmath.nstr(c[6] - c[2] * c[3], 5)}")

Both eigenforms are multiplicative, as Corollary 14.7 says, and their coefficients are conjugate elements of $\mathbb Q(\sqrt{144169})$: Galois conjugation permutes the eigenforms. (Maeda's conjecture predicts that for every $k$ the characteristic polynomial of $T_2$ on $S_k$ is irreducible with the full symmetric group as Galois group; it has been checked far beyond $k=24$.)

### P14.3
**How to find it.** $\int_1^\infty e^{-2\pi ny}y^{s-1}\,dy=(2\pi n)^{-s}\Gamma(s,2\pi n)$, and $\Gamma(s,x)\approx x^{s-1}e^{-x}$, so the terms decay like $e^{-2\pi n}$: twenty terms are plenty.

In [ ]:
mpmath.mp.dps = 30
taus = Delta[:40]
def I(s, k=12):
    return sum(taus[n] * (2 * mpmath.pi * n)**(-s) * mpmath.gammainc(s, 2 * mpmath.pi * n) for n in range(1, 40))
def Lam(s): return I(s) + I(12 - s)              # epsilon * (-1)^(k/2) = 1
# where the Dirichlet series converges absolutely (Re s > 7), compare with it
tau_many = [0] + [int(x) for x in pari('Vec(eta(q + O(q^3000))^24)')]
s = mpmath.mpc(10.5, 2.2)
series = mpmath.gamma(s) * (2 * mpmath.pi)**(-s) * sum(tau_many[n] * mpmath.mpf(n)**(-s) for n in range(1, 3000))
print("Lambda(s) from the integral:", mpmath.nstr(Lam(s), 12), "\nfrom the Dirichlet series: ", mpmath.nstr(series, 12))
L6 = Lam(6) * (2 * mpmath.pi)**6 / mpmath.gamma(6)
print("L(Delta, 6) =", mpmath.nstr(L6, 15), "  PARI:", pari('lfun(lfunetaquo([1,24]), 6)'))
def Z(t):
    s = mpmath.mpc(6, t)
    return mpmath.re(Lam(s)) / abs(mpmath.gamma(s) * (2 * mpmath.pi)**(-s))
zeros, prev, t = [], Z(0.0), 0.0
while t < 20:
    t += 0.05; cur = Z(t)
    if prev * cur < 0: zeros.append(float(mpmath.findroot(Z, t - 0.025)))
    prev = cur
print("first zeros on Re s = 6:", [round(z, 6) for z in zeros])

The integral formula, which is symmetric under $s\\mapsto12-s$ by construction, agrees with the Dirichlet series where that converges, which is the content of Theorem 14.9; the value at the centre matches PARI, and the zeros $t=9.2224,13.9075,17.4428,19.6565$ are those of Figure 14.2. All the zeros ever computed lie on the line, as for $\zeta$.

### P14.4
**How to find it.** At $s=1$, $k=2$, $\varepsilon=-1$: $\Lambda(1)=\sqrt{11}\,I(1)+\varepsilon i^2\sqrt{11}\,I(1)=2\sqrt{11}\,I(1)$ with $I(1)=\int_{1/\sqrt{11}}^\infty f(iy)\,dy=\sum_n\frac{c_n}{2\pi n}e^{-2\pi n/\sqrt{11}}$, and $\Lambda(1)=\frac{\sqrt{11}}{2\pi}L(f,1)$.

In [ ]:
N = 10**5
c = [0] + [int(x) for x in pari(f'A = eta(q + O(q^{N+1})); Vec(q * A^2 * subst(A, q, q^11)^2)')][:N]
E = pari.ellinit([0, -1, 1, 0, 0])
bad = [int(p) for p in pari.primes([2, N]) if int(p) != 11 and c[int(p)] != int(pari.ellap(E, p))]
print("primes p < 10^5 with c_p != a_p:", bad)
print("largest |c_p| / 2 sqrt(p):", max(abs(c[int(p)]) / (2 * math.sqrt(int(p))) for p in pari.primes([2, N])))
mpmath.mp.dps = 25
L1 = 2 * sum(mpmath.mpf(c[n]) / n * mpmath.exp(-2 * mpmath.pi * n / mpmath.sqrt(11)) for n in range(1, 200))
Omega = pari('ellinit([0,-1,1,0,0]).omega[1]')
print("L(f,1) =", mpmath.nstr(L1, 18), "  PARI lfun(E,1):", pari.lfun(E, 1))
print("real period Omega =", Omega, "  L/Omega =", float(L1) / float(Omega), " (1/25 =", 1 / 25, ")")
print("torsion:", pari.elltors(E), " Tamagawa numbers:", pari.elltamagawa(E))

The coefficients agree for all $9591$ primes below $10^5$ other than $11$, and $|c_p|\le2\sqrt p$ (Hasse, or the Riemann hypothesis of Chapter 13, applied to the curve). $L(f,1)=0.25384\ldots$, and $L(f,1)/\Omega=0.04=\frac1{25}$ exactly to the precision computed: with $E(\mathbb Q)=\mathbb Z/5$, Tamagawa number $1$ and Ш trivial, this is the Birch–Swinnerton-Dyer formula for a curve of rank $0$. (For this curve the rank-$0$ part is a theorem, by Kolyvagin and Gross–Zagier; the Ш part as well, by the work of Kato and Skinner–Urban.)

### P14.5
**How to find it.** With $a(k)=\frac{(6k)!}{(3k)!(k!)^3}$ the ratio of consecutive terms is a rational function of $k$: term$_k$/term$_{k-1}=-\frac{(6k-5)(2k-1)(6k-1)}{k^3\,C^3/24}$ with $C=640320$. Binary splitting returns $P,Q,T$ for a block $[a,b)$ with $P/Q$ the product of the ratios and $T/Q$ the block's contribution; two blocks combine by $P=P_1P_2$, $Q=Q_1Q_2$, $T=T_1Q_2+P_1T_2$.

In [ ]:
C = 640320; C3_24 = C**3 // 24
def bs(a, b):
    if b - a == 1:
        if a == 0: P = Q = 1
        else:
            P = (6 * a - 5) * (2 * a - 1) * (6 * a - 1)
            Q = a * a * a * C3_24
        T = P * (13591409 + 545140134 * a)
        return P, Q, -T if a & 1 else T
    m = (a + b) // 2
    P1, Q1, T1 = bs(a, m); P2, Q2, T2 = bs(m, b)
    return P1 * P2, Q1 * Q2, T1 * Q2 + P1 * T2

def chudnovsky(digits):
    n = digits // 14 + 2
    P, Q, T = bs(0, n)
    one = 10**digits
    sqrtC = math.isqrt(10005 * one * one)          # sqrt(10005) * 10^digits
    return (Q * 426880 * sqrtC) // T              # pi * 10^digits

t = time.time(); pi_bs = chudnovsky(100000); dt = time.time() - t
mpmath.mp.dps = 100010
ref = int(mpmath.floor(mpmath.pi * mpmath.mpf(10)**100000))
diff = abs(pi_bs - ref)
print(f"10^5 digits in {dt:.2f}s; difference from mpmath's value, in units of 10^-100000: {diff}")
print("pi =", str(pi_bs)[:1] + "." + str(pi_bs)[1:51] + "...")
mpmath.mp.dps = 50
x = mpmath.exp(mpmath.pi * mpmath.sqrt(163))
print("e^(pi sqrt 163) =", mpmath.nstr(x, 40), "\n640320^3 + 744   =", 640320**3 + 744, "  difference:", mpmath.nstr(x - (640320**3 + 744), 5))

The identity $12\cdot\frac{C^{3/2}}{\ldots}$ was rewritten as $\pi=\frac{426880\sqrt{10005}\,Q}{T}$, since $C^{3/2}/12=426880\sqrt{10005}$. Binary splitting makes the computation of $10^5$ digits take a fraction of a second, most of it in the final square root and division; the same algorithm (with FFT multiplication and careful memory management) produced the record computations of $\pi$ to tens of trillions of digits. The modular input is a single number, $j\bigl(\frac{1+\sqrt{-163}}2\bigr)=-640320^3$: the field $\mathbb Q(\sqrt{-163})$ has class number $1$ (Chapter 3), so this value of $j$ is an integer (Remark 14.14), and $e^{\pi\sqrt{163}}$ misses $640320^3+744$ by about $7.5\cdot10^{-13}$.